# IA102 — Apprentissage profond avec PyTorch

## Notebook 0 — Révision de IA101 et motivation

**Calcul Québec** · Atelier en ligne · Durée de ce notebook : 15 minutes

L'atelier présente les fondamentaux de l'apprentissage profond : construire des réseaux de neurones capables d'apprendre automatiquement à partir de données brutes, et les entraîner avec PyTorch. Il suit le programme annoncé :

1. **Révision de IA101** (ce notebook)
2. **Introduction aux réseaux de neurones** (notebook 1)
3. **Entraînement de réseaux de neurones avec PyTorch** (notebooks 2, 3 et 4)
4. **Introduction à l'entraînement haute performance** (notebook 5)

### Objectifs de l'atelier

| # | Objectif annoncé | À la fin de l'atelier, les participants seront en mesure de… | Notebooks |
| --- | --- | --- | --- |
| O1 | Comprendre les bases de l'apprentissage profond | expliquer le neurone artificiel, les fonctions d'activation, la rétropropagation, et la raison d'être des réseaux profonds | 0, 1 |
| O2 | Créer des réseaux de neurones | définir un réseau avec `nn.Sequential` ou `nn.Module`, choisir la couche de sortie et la fonction de perte, construire un réseau convolutif | 1, 3 |
| O3 | Entraîner des modèles avec PyTorch | écrire la boucle d'entraînement, utiliser `Dataset` et `DataLoader`, suivre et améliorer la généralisation (régularisation, transfert d'apprentissage) | 2, 3, 4 |
| O4 | S'initier à l'entraînement haute performance | utiliser un GPU, optimiser le chargement des données, employer la précision mixte et les points de contrôle, lancer un entraînement multi-GPU et le soumettre avec Slurm | 3, 5 |

### Prérequis

Statistiques, programmation Python, analyse de données avec NumPy, pandas et Matplotlib, et bases de l'apprentissage machine (IA101, PYT101, PYT102, DAT201, DAT203 ou équivalent).

### Déroulement (13 h 30 à 16 h 30)

| Heure | Notebook | Partie du programme | Contenu |
| --- | --- | --- | --- |
| 13 h 30 | 0 | Révision de IA101 | Cycle d'apprentissage, généralisation ; limites des modèles linéaires |
| 13 h 45 | 1 | Introduction aux réseaux de neurones | Neurone, activations, tenseurs, autograd, rétropropagation, `nn.Module` |
| 14 h 15 | 2 | Entraînement avec PyTorch | Boucle d'entraînement, taux d'apprentissage, optimiseurs |
| 14 h 45 | *Pause* | | |
| 14 h 55 | 3 | Entraînement avec PyTorch | Réseaux convolutifs, `DataLoader`, mini-lots, GPU |
| 15 h 30 | 4 | Entraînement avec PyTorch | Surapprentissage, régularisation, transfert d'apprentissage |
| 15 h 50 | 5 | Entraînement haute performance | Débit, chargement des données, précision mixte, points de contrôle, multi-GPU, Slurm |
| 16 h 20 | Synthèse | | Test de positionnement final, questions |

Le notebook **bonus** (Transformers et mécanisme d'attention) est proposé en étude autonome.

### Conventions

| Mention | Signification |
| --- | --- |
| **Prédiction** | Formuler une hypothèse avant d'exécuter la cellule. |
| **Exercice** | Compléter les lignes marquées `# À COMPLÉTER` ; une solution repliable suit. |
| **Question de vérification** | Question à choix multiples avec rétroaction immédiate. |
| **À retenir** | Synthèse du notebook. |

### Test de positionnement (2 minutes)

Répondez brièvement dans la cellule suivante. Le même test est repris à la fin de l'atelier (notebook 5).

1. Que calcule un neurone artificiel à partir de ses entrées ?
2. Pourquoi une fonction d'activation non linéaire est-elle nécessaire ?
3. Citez, dans l'ordre, les étapes de la boucle d'entraînement PyTorch.
4. Pourquoi un réseau convolutif convient-il mieux aux images qu'un réseau entièrement connecté ?
5. Citez deux moyens d'accélérer un entraînement sur une grappe de calcul.

In [ ]:
# Réponses au début de l'atelier
# 1.
# 2.
# 3.
# 4.
# 5.

In [ ]:
# Outil de questions de vérification (exécutez cette cellule sans la modifier)
import ipywidgets as widgets
from IPython.display import display, Markdown

def quiz(question, options, bonne, explication):
    """Affiche une question à choix multiples avec rétroaction immédiate."""
    choix = widgets.RadioButtons(options=options, value=None,
                                 layout={"width": "max-content"})
    bouton = widgets.Button(description="Vérifier", button_style="primary")
    sortie = widgets.Output()

    def verifier(_):
        with sortie:
            sortie.clear_output()
            if choix.value is None:
                display(Markdown("Veuillez choisir une réponse."))
            elif choix.value == options[bonne]:
                display(Markdown(f"**Bonne réponse.** {explication}"))
            else:
                display(Markdown("**Réponse incorrecte.** Relisez la section précédente et essayez de nouveau."))

    bouton.on_click(verifier)
    display(Markdown(f"**Question de vérification.** {question}"), choix, bouton, sortie)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

print("Version de PyTorch :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())

## 1. Révision de IA101

### Le cycle d'apprentissage

| Étape | IA101 (scikit-learn) | IA102 (PyTorch) |
| --- | --- | --- |
| **Modèle** | Fonction paramétrée, par exemple $\hat{y} = a\,x + b$ | Réseau de neurones : des milliers à des millions de paramètres |
| **Perte** | EQM (régression), entropie croisée (classification) | Les mêmes : `nn.MSELoss`, `nn.CrossEntropyLoss` |
| **Optimisation** | Cachée dans `.fit()` | Écrite explicitement : boucle de descente de gradient |
| **Gradient** | Dérivé à la main pour l'EQM : $\partial L / \partial a = \frac{2}{n}\sum (\hat{y}_i - y_i)\,x_i$ | Calculé automatiquement par **autograd** (rétropropagation) |
| **Évaluation** | Ensemble de test, validation croisée | Ensembles de validation et de test |

### La règle de mise à jour

$$\theta \leftarrow \theta - \eta\,\nabla_{\theta} L(\theta)$$

où $\theta$ désigne l'ensemble des paramètres et $\eta$ le taux d'apprentissage. Un taux trop petit ralentit la convergence ; un taux trop grand la fait diverger.

### La généralisation

| Situation | Entraînement | Validation | Action |
| --- | --- | --- | --- |
| Sous-apprentissage | Faible | Faible | Modèle plus expressif, plus d'itérations |
| Surapprentissage | Élevée | Nettement plus faible | Régularisation, plus de données, arrêt précoce |
| Bon ajustement | Élevée | Proche de l'entraînement | — |

Règle méthodologique : les réglages se choisissent sur la validation ; l'ensemble de test ne sert qu'à l'évaluation finale.

In [ ]:
quiz("Dans la règle θ ← θ − η ∇L(θ), pourquoi soustrait-on le gradient ?",
     ["Parce que le gradient indique la direction de plus forte augmentation de la perte",
      "Pour augmenter la perte", "Pour normaliser les paramètres"],
     0, "On se déplace dans la direction opposée au gradient pour faire diminuer la perte.")

In [ ]:
quiz("Exactitude d'entraînement : 99 %, exactitude de validation : 72 %. Quelle action est la plus appropriée ?",
     ["Augmenter la taille du réseau", "Régulariser ou ajouter des données", "Évaluer sur l'ensemble de test pour choisir les réglages"],
     1, "L'écart indique un surapprentissage : régularisation, augmentation de données, arrêt précoce.")

## 2. Motivation : les limites des modèles linéaires

Deux groupes de points forment des **cercles concentriques**.

**Prédiction.** Une régression logistique trace une frontière linéaire. Quelle exactitude obtiendra-t-elle ?

In [ ]:
X, y = make_circles(n_samples=600, noise=0.08, factor=0.45, random_state=0)
logreg = LogisticRegression().fit(X, y)
print(f"Exactitude de la régression logistique : {logreg.score(X, y):.0%}")

Environ 50 %, soit le hasard : aucune droite ne sépare un disque de l'anneau qui l'entoure. En IA101, il aurait fallu **construire à la main** une variable adaptée (par exemple $x_1^2 + x_2^2$). Un réseau de neurones **apprend lui-même** les représentations utiles.

Le réseau ci-dessous comporte deux couches cachées de 16 neurones. Son fonctionnement est détaillé dans les notebooks 1 et 2.

In [ ]:
torch.manual_seed(0)
Xt = torch.tensor(X, dtype=torch.float32)
yt = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

reseau = torch.nn.Sequential(torch.nn.Linear(2, 16), torch.nn.ReLU(),
                             torch.nn.Linear(16, 16), torch.nn.ReLU(),
                             torch.nn.Linear(16, 1))
optimiseur = torch.optim.Adam(reseau.parameters(), lr=0.03)
perte_fn = torch.nn.BCEWithLogitsLoss()
for epoque in range(300):
    optimiseur.zero_grad()
    perte = perte_fn(reseau(Xt), yt)
    perte.backward()
    optimiseur.step()

with torch.no_grad():
    exactitude = ((reseau(Xt) > 0).float() == yt).float().mean().item()
print(f"Exactitude du réseau de neurones : {exactitude:.0%}")

In [ ]:
xx, yy = np.meshgrid(np.linspace(-1.4, 1.4, 300), np.linspace(-1.4, 1.4, 300))
grille = np.c_[xx.ravel(), yy.ravel()]
with torch.no_grad():
    z_reseau = (reseau(torch.tensor(grille, dtype=torch.float32)) > 0).numpy().reshape(xx.shape)
z_logreg = logreg.predict(grille).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
for ax, z, titre in [(axes[0], z_logreg, f"Régression logistique : {logreg.score(X, y):.0%}"),
                     (axes[1], z_reseau, f"Réseau de neurones : {exactitude:.0%}")]:
    ax.contourf(xx, yy, z, alpha=0.25, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=12, edgecolor="k", linewidth=0.3)
    ax.set_title(titre, fontweight="bold"); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Frontière linéaire et frontière apprise par un réseau", y=1.02); plt.show()

### Quand recourir à l'apprentissage profond ?

| Critère | Apprentissage machine classique (IA101) | Apprentissage profond (IA102) |
| --- | --- | --- |
| Type de données | Tableaux de variables bien définies | Données brutes : images, signaux, textes, séquences |
| Variables | Construites par l'analyste | **Apprises** par le réseau |
| Volume de données | Centaines à milliers d'observations | Milliers à millions d'observations |
| Matériel | Un processeur suffit | **GPU** fortement recommandé |
| Contrôle de l'entraînement | Encapsulé dans `.fit()` | Boucle écrite par l'utilisateur, entièrement paramétrable |

Pour des données tabulaires de taille modeste, les méthodes d'IA101 (forêts aléatoires, régression régularisée) restent souvent le meilleur choix.

## À retenir

- Le cycle **modèle → perte → optimisation → évaluation** est le même qu'en IA101 ; PyTorch rend l'optimisation explicite et calcule les gradients automatiquement.
- Les modèles linéaires ne produisent que des frontières linéaires ; un réseau de neurones apprend des représentations non linéaires.

**Notebook suivant :** 1 — Introduction aux réseaux de neurones.